# SQL — Intermediate, Session 02: LAG/LEAD / Running Totals / Set Operations / Multi-CTE

Dataset: `datasets/chinook.db`. Second intermediate notebook.

Select the **"Python (SCLT tutor venv)"** kernel before running anything.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../../../datasets/chinook.db")

def run(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

---
## Q1 — Window function: LAG()

From `Invoice`, for rows where `CustomerId = 1`, return `InvoiceId`, `CustomerId`,
`InvoiceDate`, `Total`, and a `PrevTotal` column computed as `LAG(Total) OVER
(PARTITION BY CustomerId ORDER BY InvoiceDate)` — the previous invoice's `Total` for
that customer (the first row's `PrevTotal` should be `NULL`, since there's no prior
invoice). Order the final result by `InvoiceDate`.

In [7]:
query = """
-- your SQL here
WITH lag AS(
    SELECT
        LAG(Total) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate) AS PrevTotal,
        InvoiceId,
        CustomerId,
        InvoiceDate,
        Total
    FROM 
        Invoice
    )
SELECT
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total,
    PrevTotal
FROM
    lag
WHERE
    CustomerId = 1
ORDER BY
    InvoiceDate
"""
run(query)

,InvoiceId,CustomerId,InvoiceDate,Total,PrevTotal
0,98,1,2022-03-11 00:00:00,3.98,NaN
1,121,1,2022-06-13 00:00:00,3.96,3.98
2,143,1,2022-09-15 00:00:00,5.94,3.96
3,195,1,2023-05-06 00:00:00,0.99,5.94
4,316,1,2024-10-27 00:00:00,1.98,0.99
5,327,1,2024-12-07 00:00:00,13.86,1.98
6,382,1,2025-08-07 00:00:00,8.91,13.86


---
## Q2 — Window function: running total

Same base as Q1 (`Invoice`, `CustomerId = 1`), but instead compute `RunningTotal` as
`SUM(Total) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate ROWS BETWEEN UNBOUNDED
PRECEDING AND CURRENT ROW)` — the cumulative sum of `Total` up to and including each
row, in date order. Return `InvoiceId`, `CustomerId`, `InvoiceDate`, `Total`,
`RunningTotal`, ordered by `InvoiceDate`.

In [8]:
query = """
-- your SQL here
WITH running_total AS(
    SELECT
        SUM(Total) OVER (PARTITION BY CustomerId ORDER BY InvoiceDate ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS RunningTotal,
        InvoiceId,
        CustomerId,
        InvoiceDate,
        Total
    FROM 
        Invoice
    )
SELECT
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total,
    RunningTotal
FROM
    running_total
WHERE
    CustomerId = 1
ORDER BY
    InvoiceDate
"""
run(query)

,InvoiceId,CustomerId,InvoiceDate,Total,RunningTotal
0,98,1,2022-03-11 00:00:00,3.98,3.98
1,121,1,2022-06-13 00:00:00,3.96,7.94
2,143,1,2022-09-15 00:00:00,5.94,13.88
3,195,1,2023-05-06 00:00:00,0.99,14.87
4,316,1,2024-10-27 00:00:00,1.98,16.85
5,327,1,2024-12-07 00:00:00,13.86,30.71
6,382,1,2025-08-07 00:00:00,8.91,39.62


---
## Q3 — UNION

Return a combined list of everyone from Canada: `FirstName`, `LastName`, and a literal
third column `PersonType` — `'Customer'` for rows selected from `Customer` (`WHERE
Country = 'Canada'`), `'Employee'` for rows selected from `Employee` (`WHERE Country =
'Canada'`), combined with `UNION` (not `UNION ALL` — there shouldn't be exact duplicate
rows here, but use `UNION` to be explicit about de-duplication). Order the final result
by `LastName`.

In [ ]:
query = """
-- your SQL here
    SELECT
        FirstName,
        LastName,
        'Customer' AS PersonType
    FROM
        Customer
    WHERE
        Country = 'Canada'
UNION
    SELECT
        FirstName,
        LastName,
        'Employee' AS PersonType
    FROM
        Employee
    WHERE
        Country = 'Canada'
ORDER BY
    LastName
"""
run(query)

,FirstName,LastName,PersonType
0,Andrew,Adams,Employee
1,Robert,Brown,Customer
2,Laura,Callahan,Employee
3,Nancy,Edwards,Employee
4,Edward,Francis,Customer
5,Steve,Johnson,Employee
6,Robert,King,Employee
7,Aaron,Mitchell,Customer
8,Michael,Mitchell,Employee
9,Margaret,Park,Employee


---
## Q4 — Set difference via NOT IN

Find artists who have **no albums**: from `Artist`, return `ArtistId` and `Name` for
rows where `ArtistId` is `NOT IN (SELECT DISTINCT ArtistId FROM Album)`. Order by
`ArtistId`, limit 10.

In [20]:
query = """
-- your SQL here
SELECT
    ArtistId,
    Name
FROM
    Artist
WHERE
    ArtistId NOT IN (SELECT DISTINCT ArtistId FROM Album)
ORDER BY
    ArtistId
LIMIT 10
"""
run(query)

,ArtistId,Name
0,25,Milton Nascimento & Bebeto
1,26,Azymuth
2,28,João Gilberto
3,29,Bebel Gilberto
4,30,Jorge Vercilo
5,31,Baby Consuelo
6,32,Ney Matogrosso
7,33,Luiz Melodia
8,34,Nando Reis
9,35,Pedro Luís & A Parede


---
## Q5 — Multi-CTE

Chain two CTEs: first, `GenreCounts` — from `Track`, `GenreId` and `NumTracks`
(`COUNT(*)`), grouped by `GenreId`. Second, `TopGenres` — select from `GenreCounts`
only rows where `NumTracks > 100`. In the final query, `JOIN` `TopGenres` to `Genre`
on `GenreId`, returning `Genre.Name` as `GenreName` and `NumTracks`, ordered by
`NumTracks` descending.

In [28]:
query = """
-- your SQL here
WITH GenreCounts AS(
    SELECT
        GenreId,
        COUNT(*) AS NumTracks
    FROM
        Track
    GROUP BY
        GenreId 
),
TopGenres AS(
    SELECT
        GenreId,
        NumTracks
    FROM
        GenreCounts
    WHERE
        NumTracks > 100
)
SELECT
    t1.GenreId,
    t1.NumTracks,
    t2.Name AS GenreName
FROM
    TopGenres t1
JOIN
    Genre t2
ON
    t1.GenreId = t2.GenreId
ORDER BY
    t1.NumTracks DESC
"""
run(query)

,GenreId,NumTracks,GenreName
0,1,1297,Rock
1,7,579,Latin
2,3,374,Metal
3,4,332,Alternative & Punk
4,2,130,Jazz


---
## Done?

Say "done" in chat once all 5 run and look right.